# Огляд даних і правила очищення

Ноутбук описує якість початкових даних і обґрунтовує правила підготовки. Моделі не навчаються, вихідні таблиці не записуються; експорт виконує `02_prepare_data.ipynb`.

Кожен рядок відповідає спору, за який продавець боровся і отримав остаточний результат. `won` дорівнює 1 у разі виграшу і 0 у разі програшу. Дані не охоплюють спори, за які продавець не боровся.

In [ ]:

import pandas as pd
from IPython.display import display

from disputes.paths import DATASET
from disputes.preparation import fit_scale_reference, normalize_source, split_by_time

if not DATASET.exists():
    raise FileNotFoundError("Покладіть dataset.csv у корінь репозиторію")
raw = pd.read_csv(DATASET)

## Початковий склад

Повний дублікат — повтор усіх полів, включно з ID; такі рядки видаляються один раз. Повтор ID з різними значеннями полів не розв'язується автоматично: обробка зупиняється з помилкою.

In [ ]:
display(pd.Series({
    "рядків": len(raw),
    "колонок": len(raw.columns),
    "повних_дублікатів": int(raw.duplicated().sum()),
    "повторних_ID": int(raw.dispute_id.duplicated().sum()),
}))
display(raw.won.value_counts(dropna=False).rename("спорів").to_frame())

## Категорії й типи

Крайні пробіли видаляються; порожній рядок, `-` та `?` позначають пропуск. Варіанти `pay pal`, `mastercrad`, `mc` і різний регістр зводяться до єдиних значень. `reason_code` зберігається як текст, оскільки значення на кшталт `10.4` не є числовими величинами.

Для PayPal пропущений бренд картки замінюється на `not_card` відповідно до словника даних. Для карткових оплат пропущений бренд не відновлюється. Різні коди причин не об'єднуються на підставі спільної категорії.

In [ ]:
normalized, source_audit = normalize_source(raw)
display(pd.DataFrame({
    "тип": normalized.dtypes.astype(str),
    "пропусків": normalized.isna().sum(),
    "унікальних": normalized.nunique(),
}))
display(pd.crosstab(
    normalized.payment_method.fillna("MISSING"),
    normalized.card_brand.fillna("MISSING"),
))

In [ ]:
for column in ["psp", "reason_category", "reason_code", "currency"]:
    display(normalized[column].value_counts(dropna=False).rename("спорів").to_frame())

## Валюта `$`

`US$` трактується як USD, `€` — як EUR. Символ `$` неоднозначний; у цьому наборі він трактується як USD, оскільки в усіх парах із відомими обома сумами значення збігаються (таблиця нижче). Для решти рядків це припущення, яке фіксує прапорець `currency_dollar_assumed_usd`.

`OTHER` позначає рідкісну валюту з прихованою оригінальною сумою, тому відсутність `amount` для неї очікувана. Відома додатна сума в USD використовується без визначення валюти.

In [ ]:
dollar = source_audit.currency_dollar_assumed_usd
pairs = normalized.loc[dollar, ["amount", "amount_usd"]].dropna()
display(pd.Series({
    "позначень_долара": int(dollar.sum()),
    "відомих_пар": len(pairs),
    "пар_з_однаковими_сумами": int(pairs.amount.sub(pairs.amount_usd).abs().le(.01).sum()),
    "відновлених_not_card": int(source_audit.card_brand_restored.sum()),
    "конфліктів_оплати_та_бренду": int(source_audit.payment_brand_conflict.sum()),
}))

## Суми до виправлення

Нульові, від'ємні суми та службові значення `9 999 999` / `999 999` не вважаються достовірними сумами спору і замінюються пропусками. Знак міг мати бухгалтерський зміст, а походження службових значень невідоме, тому модуль від'ємного числа не використовується.

Рядки не видаляються; початкові значення зберігаються в аудиті. Квантилі `amount` обчислено по різних валютах, тому вони не описують єдину грошову шкалу.

In [ ]:
display(normalized[["amount", "amount_usd"]].describe(percentiles=[.25, .5, .75, .95, .99]).T)
source_flags = [column for column in source_audit if column.startswith("source__amount")]
display(source_audit[source_flags].sum().rename("спорів").to_frame())

## Часовий поділ

День тижня має бути цілим числом 0–6, тиждень — невід'ємним цілим числом; значення `999` і `2026` трактуються як службові. Рядки з некоректним або невідомим тижнем утворюють окрему частину `unknown_time`.

Train — ранній період, на якому визначаються статистичні правила; validation — наступний період для порівняння рішень; test — найпізніший період для підсумкової оцінки. Поділ не залежить від навчання моделі.

Загальна статистика й результати за тижнями переглядалися до поділу, тому test не є повністю непереглянутим. Правила за test не підбираються.

In [ ]:
parts = split_by_time(normalized)
display(pd.DataFrame([
    {"частина": name, "рядків": len(part),
     "перший_тиждень": part.dispute_week.min(), "останній_тиждень": part.dispute_week.max()}
    for name, part in parts.items()
]))

## Помилка масштабу суми в USD

Опорні співвідношення `amount_usd / amount` визначаються лише на train: додатні скінченні пари без службових значень, обидві суми не менші за 10, не менше 30 пар на валюту. Межі — 5-й і 95-й процентилі із запасом 5 %.

Сума в USD ділиться на 1000 лише тоді, коли початкове співвідношення перевищує медіану валюти щонайменше в 100 разів, а виправлене співвідношення потрапляє в межі. Для USD додатково вимагається збіг двох сум із допуском 0,01.

Правило стосується однієї конкретної аномалії. Воно не доводить, що помилка міститься саме в колонці USD, не виправляє інших масштабів і не замінює довідника валютних курсів. Валюти з недостатньою кількістю пар (зокрема PLN) правилом не охоплюються.

In [ ]:
reference = fit_scale_reference(parts["train"])
display(reference[["pairs", "median", "lower", "upper"]])

## Відновлення та невирішені суми

Для валюти USD відсутня сума відновлюється з іншої, якщо джерело додатне і не перевищує 10 000. Для інших валют відновлення не виконується.

Після виправлення й відновлення сума понад 10 000 зберігається лише в аудиті, а в основній таблиці замінюється пропуском; для `amount` поріг задано в оригінальній валюті. За розбіжності двох сум у USD обидві замінюються пропусками. Поріг є робочим правилом перевірки, а не встановленим максимумом платежу.

Аудиторські колонки фіксують походження суми: спостережена, виправлена за масштабом, відновлена, недоступна або така, що потребує перевірки.

## Межі етапу

Пропуски зберігаються. Заповнення пропусків, кодування категорій, масштабування ознак і вибір моделі виконуються на наступних етапах і налаштовуються лише на train. Початкові суми, ID, ціль і прапорці оцінювання не входять до ознак моделі.

Вибірка містить лише спори, за які боролися й отримали результат. Дата отримання результату відсутня, тому не встановлено, що всі мітки ранніх спорів були відомі на початок наступного періоду. Очищення цього обмеження не усуває.